# 壁運動 / asynergy 表記の探索（読み取り専用）

- DBは `mode=ro` で開く。書き込みなし。
- 全角・半角はNFKCで統一し、小文字化して検索する。
- 目的: 実際の表記と頻度を把握し、抽出ルールを決める材料にする。

In [1]:
import sqlite3, re, unicodedata
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"

conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
df = pd.read_sql(
    "SELECT Study_ID, visit_datetime, record_no, karte_text "
    "FROM karte WHERE karte_text IS NOT NULL",
    conn
)
conn.close()
print("loaded:", len(df))

loaded: 438612


In [2]:
# キーワード: asynergy系の綴りゆれ、dyssynergy、アシナジー、壁運動
KW = re.compile(r"asyn\w*|dyssyn\w*|アシナジー|壁運動", re.I)
WIN = 30  # 前後の文字数

def norm(s):
    return unicodedata.normalize("NFKC", str(s)).lower()

rows = []
for r in df.itertuples(index=False):
    t = norm(r.karte_text)
    for m in KW.finditer(t):
        rows.append({
            "Study_ID": r.Study_ID,
            "visit_datetime": r.visit_datetime,
            "record_no": r.record_no,
            "keyword": m.group(0),
            "before": t[max(0, m.start() - WIN):m.start()].replace("\n", "⏎"),
            "after": t[m.end():m.end() + WIN].replace("\n", "⏎"),
        })

hits = pd.DataFrame(rows)
n_rec = hits[["Study_ID", "visit_datetime", "record_no"]].drop_duplicates().shape[0]
print("hits:", len(hits), "/ records:", n_rec, "/ all records:", len(df))

hits: 46482 / records: 24127 / all records: 438612


In [3]:
# 1) ヒットしたキーワードの綴り別頻度
hits["keyword"].value_counts()

keyword
asynergy                                     40178
壁運動                                           5072
asynergyなし                                     436
dyssynchrony                                   376
asynegy                                        110
asyneryg                                        58
asynergyはっきりせず                                  40
dyssynchronyあり                                  26
asynergyはなさそうです                                 24
asynergyはなさそう                                   23
asyner                                          23
asynergyの明らかな変化無くその他も著変なし                       23
dyssynchronyの影響                                 14
アシナジー                                           14
asynergydd                                      11
asynergyや弁膜異常なし                                  9
asynchronyが原因か右冠動脈と左冠動脈位相がmatchせずそれぞれで再構成        9
asynrgy                                          4
asynerygy                                        2
dyssynchrony著明         

In [4]:
# 2) キーワード直後12文字のパターン頻度（数字は # に置換）
after12 = (
    hits["after"].str[:12]
    .str.replace(r"\d+", "#", regex=True)
)
pat = pd.DataFrame({"keyword": hits["keyword"], "after12": after12})
pat.value_counts().head(100).rename("n").reset_index()

,keyword,after12,n
0,asynergy,", ef #%, iv",22471
1,asynergy,",ef #%,ivst",3060
2,asynergy,"-, ivst #mm",2664
3,asynergy,"-, ivst #mm,",2131
4,asynergy,", ef#%, ivs",1629
5,asynergy,"-, ef #%, i",813
6,壁運動,"なし, ivst #m",704
7,asynergy,"-,ivst #mm,",684
8,壁運動,"低下なし, ivst #",494
9,asynergy,"-, aod #mm,",492


In [5]:
# 3) 直後の表現を大まかに分類（探索用。抽出ルールではない）
DASH = r"[-−‐‑–—―ー]"
groups = {
    "dash(-)":   rf"^\s*[\(\[]?\s*{DASH}\s*[\)\]]?",
    "なし/無し":   r"^\s*(は|も)?\s*[\(\[]?\s*(異常)?\s*(なし|無し|無|認めず|認めない)",
    "異常なし":    r"^\s*異常\s*(なし|無し|無|[\(\[]?" + DASH + r"[\)\]]?)",
    "あり/(+)":   r"^\s*(は|も)?\s*[\(\[]?\s*(異常)?\s*(あり|有り|有|\+|＋)",
    "低下/無収縮": r"低下|無収縮|akinesis|hypokinesis|dyskinesis",
}

def label(a):
    hit = [k for k, p in groups.items() if re.search(p, a)]
    return "|".join(hit) if hit else "other"

hits["label"] = hits["after"].map(label)
hits["label"].value_counts()

label
other                31609
dash(-)               9941
低下/無収縮                2796
なし/無し                 1238
あり/(+)                 452
なし/無し|異常なし             404
dash(-)|低下/無収縮          25
あり/(+)|低下/無収縮           13
異常なし                     3
なし/無し|異常なし|低下/無収縮        1
Name: count, dtype: int64

In [6]:
# 4) 分類ごとのサンプル（各8件）。other は特に確認する
for lab, g in hits.groupby("label"):
    print(f"\n=== {lab} (n={len(g)}) ===")
    for r in g.sample(min(8, len(g)), random_state=0).itertuples():
        print(f"[{r.Study_ID}] ...{r.before}【{r.keyword}】{r.after}")


=== dash(-) (n=9941) ===
[P105319] ...臓超音波検査:dd 45mm,ds mm, ef 71%, 【asynergy】-, ivst 12mm, pwt 12mm, aod 30
[P703162] ...13mm, aod 27.3mm, lad 36.5mm, 【asynergy】- ef 67%,e/e’12, mild ar, trac
[P672282] ...音波検査:dd 34mm,ds 19mm, ef 78%, 【asynergy】-, ivst 12mm, pwt 11mm, aod 35
[P081944] ..., ds 32mm, ivst 9mm, pwt 9mm, 【asynergy】-, aod 34mm, lad 34mm, lavol 3
[P062225] ...年11月心臓超音波検査:dd 43mm, ds 30mm, 【asynergy】-, ef 58%, ivst 10mm, pwt 9mm,
⏎2022年7月心臓超音波検査:dd 38mm, 【asynergy】-, lvh-, ef 70%, aod 28mm, lad
[P352771] ...ds 29mm, ivst 10mm, pwr 10mm, 【asynergy】-, ef 66%, lad 43mm, aod 27mm,
[P677996] ... ds 27mm, ivst 10mm, pwt 8mm, 【asynergy】-, aod 32mm, lad 39mm, lav 63m

=== dash(-)|低下/無収縮 (n=25) ===
⏎心臓超音波検査:dd 40mm, 【asynrgy】-, 1mr, 1tr, 1as(無冠尖左冠尖の動き低下あり
⏎P008018] ...:dd 40mm, ivst 10mm, pwt 9mm, 【asynergy】-, 2as(石灰化可動性低下あり)、ivc 描出困難。
⏎、右心拡大なし、trなしdd 43mm, 【asynergy】-, 大動脈弁可動性低下2度as
⏎、右心拡大なし、trなしdd 43mm, 【asynergy】-, 大動脈弁可動性低下2度as
⏎初診時心臓胸部超音波:dd 35mm, lvh-, 【asynergy】-, 1as (左冠尖石灰化

In [7]:
# 5) 同一記録に壁運動所見とEF数値が併記されている割合（補完対象の規模確認）
EF_ANY = re.compile(r"ef\s*[:=(（]?[^%％\n]{0,12}?\d+(?:\.\d+)?\s*[%％]", re.I)

key = ["Study_ID", "visit_datetime", "record_no"]
rec_hit = hits[key].drop_duplicates()

tmp = df.merge(rec_hit.assign(has_wm=1), on=key, how="left")
tmp["has_wm"] = tmp["has_wm"].fillna(0).astype(int)
tmp["has_ef"] = tmp["karte_text"].map(lambda s: int(bool(EF_ANY.search(norm(s)))))
pd.crosstab(tmp["has_wm"], tmp["has_ef"], margins=True)

has_ef,0,1,All
has_wm,,,
0,405196,9289,414485
1,2579,21548,24127
All,407775,30837,438612


In [8]:
# 6) 結果をCSVに保存（DBは変更しない）
hits.to_csv("wall_motion_hits.csv", index=False, encoding="utf-8-sig")
print("saved: wall_motion_hits.csv", len(hits))

saved: wall_motion_hits.csv 46482
